In [1]:
import cProfile
import pandas as pd
from src.recovery_model import RecoveryModel
import os

pd.set_option("multi_sparse", False)
pd.set_option("display.float_format", "{:.2f}".format)

## test with 40 flows


In [2]:
folder = "test_1"
path = f"data/{folder}/"
elt = "Ag"
folder_out = f"{folder}_{elt}"

In [3]:
# # original
# inputs = pd.read_csv(path + "inputs.csv")
# composition = pd.read_csv(path + "composition.csv")
# TCs = pd.read_csv(path + "TCs.csv")
# metadata = pd.read_csv(path + "metadata.csv")

# # Remove all but "Ag" element

# sgl_Elt = (composition["element"].isna()) | (composition["element"] == elt)
# composition = composition[sgl_Elt]

# mask = metadata["element"] == elt
# metadata["element"] = None
# metadata.loc[mask, "element"] = elt

# mask1 = (TCs["input_layer"] == "element") & (TCs["input_layer_key"] != elt)
# mask2 = (TCs["output_layer"] == "element") & (TCs["output_target_key"] != elt)
# mask = mask1 | mask2
# TCs = TCs[~mask]

# path_out = f"data/{folder_out}/"
# if not os.path.exists(path_out):
#     os.makedirs(path_out)

# inputs.to_csv(path_out + "inputs.csv", index=False)
# composition.to_csv(path_out + "composition.csv", index=False)
# TCs.to_csv(path_out + "TCs.csv", index=False)
# metadata.to_csv(path_out + "metadata.csv", index=False)

# del metadata, inputs, composition, TCs

In [4]:
# Select a folder for the data to be used
folder = "test_2"  # test_1  test_2  Toy_WEEE_v2 folder_out
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
layer_4 = "element"

In [5]:
layer_names = (layer_0, layer_1, layer_2, layer_3, layer_4)
all_symbols = {layer_1: "P*", layer_2: "C*", layer_3: "M*", layer_4: "E*"}

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Layer 4": layer_4,
        "Value": "data",
        "Year": "year",
        "Scenario": "scenario",
        "Location": "region",
        "parameterCode": "parameterCode",
        "UoM": "unit",
    },
    "parameterCode": {
        layer_2: "c-p",
        layer_3: "m-c",
        layer_4: "e-m",
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Year": "year",
        "Unit": "unit",
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
        "process": "process",
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        layer_4: "E*",
    },
}

---


# Recovery model


In [6]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

---

# Metadata


In [7]:
model.dims

(8, 3, 4, 3, 3)

In [8]:
model.solve(aggregate=False, pivot=False)

,flow,product,component,material,element,data
0,F1,P1,NaN,NaN,NaN,1000.00
1,F1,P1,C1,NaN,NaN,250.00
2,F1,P1,C1,M1,NaN,130.00
3,F1,P1,C1,M1,E1,91.00
4,F1,P1,C1,M1,E2,39.00
...,...,...,...,...,...,...
175,F8,P1,C3,M2,E2,0.40
176,F8,P2,C1,M1,E1,0.23
177,F8,P2,C1,M2,E2,0.09
178,F8,P2,C2,M1,E1,0.28


In [9]:
12_000_000 / (2 * 31 * 83 * 8)

291.4885347842985

In [10]:
295 * 8 * 83 * 31 * 2

12144560

In [11]:
model.size

np.int64(864)

In [12]:
model.flows_eqs

,F1,F2,F3,F4,F5,F6,F7,F8
process,,,,,,,,
T1,1,-1,-1,0,0,1,0,0
T2,0,1,0,-1,-1,0,0,0
T3,0,0,0,0,1,0,1,-1
T4,0,0,1,1,0,-1,-1,0


list(enumerate(model.sub_systems))


In [13]:
print(model)

{   'Layer 1': {'P*': 2, 'P1': 0, 'P2': 1},
    'Layer 2': {'C*': 3, 'C1': 0, 'C2': 1, 'C3': 2},
    'Layer 3': {'M*': 2, 'M1': 0, 'M2': 1},
    'Layer 4': {'E*': 2, 'E1': 0, 'E2': 1},
    'Stock/Flow ID': {   'F1': 0,
                         'F2': 1,
                         'F3': 2,
                         'F4': 3,
                         'F5': 4,
                         'F6': 5,
                         'F7': 6,
                         'F8': 7},
    'Substance_main_parent': {'P*': 2, 'P1': 0, 'P2': 1},
    'Unit': {'Mg': 0},
    'component': {'C*': 3, 'C1': 0, 'C2': 1, 'C3': 2},
    'element': {'E*': 2, 'E1': 0, 'E2': 1},
    'flow': {   'F1': 0,
                'F2': 1,
                'F3': 2,
                'F4': 3,
                'F5': 4,
                'F6': 5,
                'F7': 6,
                'F8': 7},
    'material': {'M*': 2, 'M1': 0, 'M2': 1},
    'parameterCode': {'c-p': 0, 'e-m': 1, 'm-c': 2},
    'process': {'T1': 0, 'T2': 1, 'T3': 3, 'T4': 2},
    'produ

In [14]:
model.unravel_index(567)

array([[ 5, -1,  2, -1, -1]])

In [15]:
model.dims

(8, 3, 4, 3, 3)

In [16]:
model.size

np.int64(864)

In [17]:
model.flows_eqs

,F1,F2,F3,F4,F5,F6,F7,F8
process,,,,,,,,
T1,1,-1,-1,0,0,1,0,0
T2,0,1,0,-1,-1,0,0,0
T3,0,0,0,0,1,0,1,-1
T4,0,0,1,1,0,-1,-1,0


In [18]:
list(enumerate(model.sub_systems))

[(0, {'T1', 'T2', 'T4'}), (1, {'T3'})]

In [19]:
print(model)

{   'Layer 1': {'P*': 2, 'P1': 0, 'P2': 1},
    'Layer 2': {'C*': 3, 'C1': 0, 'C2': 1, 'C3': 2},
    'Layer 3': {'M*': 2, 'M1': 0, 'M2': 1},
    'Layer 4': {'E*': 2, 'E1': 0, 'E2': 1},
    'Stock/Flow ID': {   'F1': 0,
                         'F2': 1,
                         'F3': 2,
                         'F4': 3,
                         'F5': 4,
                         'F6': 5,
                         'F7': 6,
                         'F8': 7},
    'Substance_main_parent': {'P*': 2, 'P1': 0, 'P2': 1},
    'Unit': {'Mg': 0},
    'component': {'C*': 3, 'C1': 0, 'C2': 1, 'C3': 2},
    'element': {'E*': 2, 'E1': 0, 'E2': 1},
    'flow': {   'F1': 0,
                'F2': 1,
                'F3': 2,
                'F4': 3,
                'F5': 4,
                'F6': 5,
                'F7': 6,
                'F8': 7},
    'material': {'M*': 2, 'M1': 0, 'M2': 1},
    'parameterCode': {'c-p': 0, 'e-m': 1, 'm-c': 2},
    'process': {'T1': 0, 'T2': 1, 'T3': 3, 'T4': 2},
    'produ

---

# Linear equations


In [20]:
model.lneqs

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 441 stored elements and shape (864, 864)>

In [21]:
model.lneqs.nanmin()

np.float64(0.0)

---

# Constant terms


In [22]:
model.y

<Compressed Sparse Column sparse array of dtype 'int64'
	with 2 stored elements and shape (864, 1)>

---

# Solver


In [23]:
solution = model.solve(aggregate=True, pivot=True)
solution.fillna("")

,flow,F1,F2,F3,F4,F6,F5,F7,F8
layer,key,,,,,,,,
product,P1,1000.00,,,,,,,
product,P2,700.00,,,,,,,
component,C1,551.00,86.58,,,,,,
component,C2,639.00,51.57,39.40,10.31,17.32,,,
component,C3,510.00,,13.04,,1.83,,,
material,M1,920.43,84.53,35.28,7.90,13.72,21.51,4.56,
material,M2,779.57,53.62,17.16,2.41,5.42,19.33,4.99,
element,E1,676.76,79.40,14.76,3.39,5.66,24.33,2.42,2.32
element,E2,1023.24,58.75,37.68,6.93,13.48,16.51,7.14,1.19


In [24]:
model.solve(aggregate=True, pivot=False)

,flow,layer,key,data
0,F1,product,P1,1000.00
1,F1,product,P2,700.00
0,F1,component,C1,551.00
1,F1,component,C2,639.00
2,F1,component,C3,510.00
3,F2,component,C1,86.58
4,F2,component,C2,51.57
5,F3,component,C2,39.40
6,F3,component,C3,13.04
7,F4,component,C2,10.31


In [25]:
model.solve(aggregate=False, pivot=True).fillna("")

flow,product,component,material,element,F1,F2,F3,F4,F5,F6,F7,F8
0,P1,,,,1000.00,,,,,,,
1,P1,C1,,,250.00,62.50,,,,,,
2,P1,C1,M1,,130.00,32.50,,,12.68,,,
3,P1,C1,M1,E1,91.00,22.75,,,8.87,,,1.60
4,P1,C1,M1,E2,39.00,9.75,,,3.80,,,
5,P1,C1,M2,,120.00,30.00,,,9.60,,,
6,P1,C1,M2,E1,117.60,29.40,,,9.41,,,
7,P1,C1,M2,E2,2.40,0.60,,,0.19,,,0.01
8,P1,C2,,,590.00,27.76,26.86,5.55,,11.24,,
9,P1,C2,M1,,389.40,18.32,17.73,3.66,1.83,7.42,2.08,


In [26]:
model.solve(aggregate=False, pivot=False).fillna("")

,flow,product,component,material,element,data
0,F1,P1,,,,1000.00
1,F1,P1,C1,,,250.00
2,F1,P1,C1,M1,,130.00
3,F1,P1,C1,M1,E1,91.00
4,F1,P1,C1,M1,E2,39.00
...,...,...,...,...,...,...
175,F8,P1,C3,M2,E2,0.40
176,F8,P2,C1,M1,E1,0.23
177,F8,P2,C1,M2,E2,0.09
178,F8,P2,C2,M1,E1,0.28


## Output restructuring


In [27]:
sol = model.solve(aggregate=False, pivot=False)

base_cols = ["flow"]  # add countries, years, etc. (if needed)
levels = ["product", "component", "material", "element"]  # to be adapted to the model
results = dict()

for i, level in enumerate(levels):

    # consider the rows where the level is not empty but the sublevels are empty
    filter_mask = (sol[level].notna()) & (sol[levels[i + 1 :]].isna().all(axis=1))

    # define the columns to drop
    cols_to_drop = [col for col in levels if col != level]
    cols_to_keep = base_cols + [level]

    # sum the data for the flows and the level (disregarding the sublevels)
    temp_df = sol.loc[filter_mask].groupby(cols_to_keep).sum()

    # save the results (drop the columns that are not of interest)
    results[level] = temp_df.drop(columns=cols_to_drop)

    # # Uncomment to display the results
    # display(results[level])

# Check the results (e.g. for elements)
results["element"]

,,data
flow,element,
F1,E1,676.76
F1,E2,1023.24
F2,E1,79.40
F2,E2,58.75
F3,E1,14.76
F3,E2,37.68
F4,E1,3.39
F4,E2,6.93
F5,E1,24.33


In [ ]:
# ---------------------- TEST
cols = ["product", "component", "material", "element", "WEEE_generatedDedicated"]
TEST_REST = model.solve(aggregate=False, pivot=True).fillna("")[cols]
TEST_REST

In [ ]:
model.dims

In [ ]:
counter = TEST_REST[cols[:-1]].apply(np.unique)

print(np.prod(counter.apply(len)))
print(counter.apply(len))

In [ ]:
temp = TEST_REST[TEST_REST["product"] == "WEEE_Cat3"]

counter = temp[cols[:-1]].apply(np.unique)

print(np.prod(counter.apply(len)))
print(counter.apply(len))

---

# Performance


In [ ]:
%%prun -l 20 -s cumulative

model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

In [ ]:
cProfile.run(
    "RecoveryModel(name=folder, metadata=metadata, composition=composition, inputs=inputs, tcs=tcs, layer_names=layer_names, save_intermediary_steps=False, save_duplicates=False)",
    "performance/RecoveryModel.pstats",
)

cProfile.run(
    "model.solve(aggregate=False, pivot=True)",
    "performance/Solver.pstats",
)

# # ! then go into results/performances/ and run the following commands
# # (-n 2 means cutoff at 2%)
# gprof2dot -f pstats -n 1 RecoveryModel.pstats | dot -Tpng -o RecoveryModel.png
# gprof2dot -f pstats -n 1 Solver.pstats | dot -Tpng -o Solver.png

In [5]:
cProfile.run(
    "RecoveryModel(name=folder, metadata=metadata, composition=composition, inputs=inputs, tcs=tcs, layer_names=layer_names, save_intermediary_steps=False, save_duplicates=False)",
    "performance/RecoveryModel.prof",
)

cProfile.run(
    "model.solve(aggregate=False, pivot=True)",
    "performance/Solver.prof",
)

# # ! then go into results/performances/ and run the following commands
# snakeviz RecoveryModel.prof
# snakeviz Solver.prof  # 22.6s

          product            component                   material element  \
8022    WEEE_Cat1                  NaN                        NaN     NaN   
8023    WEEE_Cat1  ComponentShadowWEEE                        NaN     NaN   
8024    WEEE_Cat1  ComponentShadowWEEE              AlAndAlAlloys     NaN   
8025    WEEE_Cat1  ComponentShadowWEEE              AlAndAlAlloys      Ag   
8026    WEEE_Cat1  ComponentShadowWEEE              AlAndAlAlloys      As   
...           ...                  ...                        ...     ...   
29563  WEEE_Cat4b   passiveJunctionBox              CuAndCuAlloys      Zr   
29564  WEEE_Cat4b   passiveJunctionBox  otherOrUndefinedMaterials     NaN   
29565  WEEE_Cat4b   passiveJunctionBox  otherOrUndefinedMaterials      Ag   
29566  WEEE_Cat4b   passiveJunctionBox                   plastics     NaN   
29567  WEEE_Cat4b   passiveJunctionBox                   plastics      Ag   

              process  WEEE_generatedComplementaryExported  \
8022   WEEE_g